# Alakoro FiberSense v2.2.1 — Demonstracao Interativa
# DFOS Platform for Oil & Gas Wells — Interactive Demo

**Autor/Author:** Luiz Paulo Colombiano | **Licenca/License:** MIT | **Versao/Version:** 2.2.1

---

## 1. Instalacao / Installation

```bash
pip install alakoro-fibersense
```

Ou execute esta celula / Or run this cell:

In [ ]:
# Instalar (se necessario) / Install (if needed)
!pip install alakoro-fibersense

## 2. Importar Modulos / Import Modules

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.simulation import SignatureGenerator, WellGeometry, AcquisitionConfig
from src.processing import LFDASProcessor
from src.validation import SignatureValidator

print('Alakoro FiberSense carregado! / loaded!')

## 3. Configurar o Poco / Configure the Well

In [ ]:
well = WellGeometry(depth_top=0, depth_bottom=3000, n_channels=3000)
acq = AcquisitionConfig(sampling_rate_hz=1000, trace_interval_s=2.0, duration_s=3600)

print(f'Poco / Well: {well.n_channels} canais, {well.depth_bottom}m')
print(f'Aquisicao / Acquisition: {acq.sampling_rate_hz} Hz, {acq.duration_s}s')

## 4. Gerar Assinatura Joule-Thomson / Generate Joule-Thomson Signature

In [ ]:
gen = SignatureGenerator(well, acq)
jt = gen.generate_joule_thomson(interface_depth=1500.0)

print(f'Assinatura / Signature: {jt["signature_type"].pt}')
print(f'   EN: {jt["signature_type"].en}')
print(f'   DTS shape: {jt["dts"].shape}')
print(f'   DAS shape: {jt["das"].shape}')

## 5. Visualizar / Visualize

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

depth = well.depth_array
time = np.arange(0, acq.duration_s, acq.trace_interval_s)

# Mapa de calor DTS / DTS heatmap
ax1 = axes[0]
im1 = ax1.imshow(jt["dts"].T, aspect='auto', cmap='RdBu_r',
                 extent=[0, acq.duration_s, well.depth_bottom, well.depth_top])
ax1.axhline(y=1500, color='yellow', linestyle='--', label='interface (1500m)')
ax1.set_xlabel('Tempo (s) / Time (s)')
ax1.set_ylabel('Profundidade (m) / Depth (m)')
ax1.set_title('DTS — Joule-Thomson')
ax1.legend()
ax1.invert_yaxis()
plt.colorbar(im1, ax=ax1, label='Temperatura (C)')

# Perfil em tempos selecionados / Profile at selected times
ax2 = axes[1]
for t in [0, 300, 900, 1500, 1800]:
    t_idx = int(t / acq.trace_interval_s)
    ax2.plot(depth, jt["dts"][t_idx, :], label=f't={t}s', alpha=0.7)
ax2.axvline(x=1500, color='red', linestyle='--', alpha=0.5)
ax2.set_xlabel('Profundidade (m) / Depth (m)')
ax2.set_ylabel('Temperatura (C)')
ax2.set_title('Perfis de Temperatura / Temperature Profiles')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Processar LF-DAS / Process LF-DAS

In [ ]:
lfdas = LFDASProcessor(cutoff_hz=1.0, refresh_rate_target_s=2.0)
result = lfdas.process(jt["das"], trace_interval_s=2.0)

print(f'LF-DAS processado! / processed!')
print(f'   Temperature shape: {result["temperature"].shape}')
print(f'   Refresh rate: {result["refresh_rate_s"]:.2f}s')
print(f'   Thermal coefficient: {result["metadata"]["thermal_coefficient"]}')

## 7. Validar / Validate

In [ ]:
validator = SignatureValidator(well, acq)
validation = validator.validate_signature(jt, result)

print(f'Validacao / Validation: {validation["passed"]}/{validation["total"]} passaram')
print(f'   Taxa de sucesso / Success rate: {validation["success_rate"]:.0f}%')
print()
for test in validation["tests"]:
    status = '✅' if test["passed"] else '❌'
    print(f'   {status} {test["message"]}')

## 8. Gerar Todas as 15 Assinaturas / Generate All 15 Signatures

In [ ]:
signatures = {
    'Joule-Thomson': gen.generate_joule_thomson(),
    'Slope Velocity': gen.generate_slope_velocity(),
    'Warm-Back': gen.generate_warm_back(),
    'Valve Chatter': gen.generate_valve_chatter(),
    'Slugging Cycle': gen.generate_slugging_cycle(),
    'Leak Path': gen.generate_leak_path(),
    'GLV Rupture': gen.generate_glv_bellow_rupture(),
    'Perforation': gen.generate_perforation_effectiveness(),
    'Frac Screen-out': gen.generate_frac_screenout(),
    'Proppant Dist': gen.generate_frac_proppant_distribution(),
    'Frac Height': gen.generate_frac_height_growth(),
    'Cement Bond': gen.generate_cement_bond_evaluation(),
    'Re-Cementing': gen.generate_re_cementing_assessment(),
    'Crossflow': gen.generate_crossflow_zonal(),
    'Cement Channel': gen.generate_cement_channeling(),
}

print(f'Geradas {len(signatures)} assinaturas! / Generated {len(signatures)} signatures!')
for name, sig in signatures.items():
    print(f'   {name:20s} — {sig["signature_type"].pt}')

## 9. Validar Todas / Validate All

In [ ]:
all_results, success_rate = validator.run_full_validation(signatures, lfdas)
print(f'
Taxa de sucesso media / Average success rate: {success_rate:.1f}%')

---

## Documentacao / Documentation

- [README.md](README.md) — Visao geral / Overview
- [INSTALL.md](INSTALL.md) — Guia de instalacao / Installation guide
- [USER_GUIDE.md](USER_GUIDE.md) — Guia do usuario / User guide
- [CONTRIBUTING.md](CONTRIBUTING.md) — Como contribuir / How to contribute

**Licenca / License:** MIT — Luiz Paulo Colombiano, 2026